# AMG-preconditioned Krylov unfolding (Rlinsolve) (`unfold_amg`)

This notebook applies **AMG/stationary-preconditioned Krylov unfolding** — the Python analogue of the R
package **Rlinsolve (+pyamg)** — to a realistic benchmark: detector readings
synthesised from the **Monte-Carlo calculated spectrum
`t4-14-s.txt_1`** of the
[IAEA Compendium](https://www-nds.iaea.org/benchmarks/), a BNCT-like
beam-shaping-assembly spectrum with a thermal group, an epithermal
$1/E$ region and a fast peak.

We use the built-in GSF response functions (10 Bonner spheres, `0in` –
`18in`, 60 energy bins from 1e-9 to ~631 MeV).  Detector readings are
folded with `Detector.get_effective_readings_for_spectra`, the
spectrum is reconstructed with `unfold_amg`, and the result is
compared against the ground truth — which never enters the unfolding.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bssunfold import Detector, RF_GSF
from bssunfold.utils.comparison import compare_spectra

detector = Detector(RF_GSF)
E = detector.E_MeV
names = detector.detector_names
print(f"Detector grid: {detector.n_energy_bins} bins, "
      f"{E[0]:.1e} - {E[-1]:.1f} MeV")
print("Spheres:", ", ".join(names))
detector.plot_response_functions()


## 1. IAEA Compendium reference spectrum → detector readings

The compendium CSV stores 61-point Monte-Carlo spectra on its
own energy grid; `get_effective_readings_for_spectra` folds
the spectrum with the response functions and resamples it
onto the 60-bin detector grid.

In [ ]:
reference_csv = pd.read_csv(
    '../tests/MonteCarlo_Calculated_spectra_from_IAEA_Comp_for_comparison.csv'
)
readings = detector.get_effective_readings_for_spectra(
    reference_csv[['E_MeV', 't4-14-s.txt_1']]
)
print("Effective readings:")
for nm in names:
    print(f"  {nm:>5s}: {readings[nm]:.4g}")

phi_true = np.interp(
    E, reference_csv['E_MeV'].values, reference_csv['t4-14-s.txt_1'].values
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
ax.loglog(E, phi_true, "k-", lw=1.5)
ax.set(xlabel="E, MeV", ylabel=r"$\varphi(E)$, cm$^{-2}$s$^{-1}$bin$^{-1}$",
       title="IAEA Compendium spectrum t4-14-s.txt_1 (ground truth)")
ax.grid(True, which="both", ls=":", alpha=0.5)

ax = axes[1]
vals = [readings[nm] for nm in names]
ax.bar(np.arange(len(names)), vals, color="steelblue")
ax.set_yscale("log")
ax.set_xticks(np.arange(len(names)))
ax.set_xticklabels(names, rotation=45)
ax.set(xlabel="sphere", ylabel="reading, a.u.",
       title="Effective Bonner-sphere readings")
ax.grid(True, axis="y", ls=":", alpha=0.5)
fig.tight_layout()
plt.show()


## 2. AMG-preconditioned Krylov unfolding

`unfold_amg` solves the auto-damped normal equations
`(A^T A + reg I) x = A^T b` with `cg`/`bicgstab`/`gmres` accelerated
by algebraic multigrid (smoothed aggregation, the Rlinsolve/pyamg
analogue) or by one sweep of a classical stationary iteration
(Jacobi/GS/SOR/SSOR).  Projected outer restarts keep the spectrum
non-negative.

In [ ]:
result = detector.unfold_amg(
    readings,
    method="cg",
    preconditioner="amg",
    max_iterations=200,
    tolerance=1e-10,
    save_result=False,
)

print(f"method        : {result['method']}")
print(f"iterations    : {result['iterations']}")
print(f"converged     : {result['converged']}")

lines_to_plot = [
    ("AMG-CG (pyamg; falls back to Jacobi if missing)", result['spectrum'],
     "C1-"),
]


## 3. Preconditioner comparison

Compare the AMG preconditioner with the classical stationary
iterations of Rlinsolve (Jacobi, Gauss-Seidel, SOR, SSOR) on the
same system.

In [ ]:
results_pc = {}
for pc in ["amg", "jacobi", "gs", "sor", "ssor"]:
    try:
        res = detector.unfold_amg(
            readings, method="cg" if pc not in ("gs", "sor") else "gmres",
            preconditioner=pc, save_result=False,
        )
        results_pc[pc] = res
    except Exception as exc:
        print(f"{pc:>7s}: failed - {exc}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.loglog(E, phi_true, "k-", lw=2, label="IAEA ground truth")
for pc, res in results_pc.items():
    ax.loglog(E, res['spectrum'], lw=1.1, label=f"{pc}")
ax.set(xlabel="E, MeV", ylabel=r"$\varphi(E)$, cm$^{-2}$s$^{-1}$bin$^{-1}$",
       title="AMG vs. stationary preconditioners (Rlinsolve family)")
ax.set_xlim(E[0], 30)
ax.grid(True, which="both", ls=":", alpha=0.35)
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

for pc, res in results_pc.items():
    q = compare_spectra(
        res['spectrum'], phi_true,
        metrics=['pearson_r', 'relative_flux_error',
                 'comprehensive_score'],
    )
    print(f"{pc:>7s}: iters={res['iterations']:3d}  "
          f"pearson_r={q['pearson_r']:.3f}")


## Quality assessment

`compare_spectra` reports the reconstruction metrics against the
independently known IAEA Compendium spectrum (used only for
evaluation).

In [ ]:
quality = compare_spectra(
    result['spectrum'], phi_true,
    metrics=["relative_flux_error", "pearson_r", "comprehensive_score",
             "fluence_difference_percent", "dose_difference_percent"],
    energy=E,
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.loglog(E, phi_true, "k-", lw=2, label="IAEA ground truth")
for label, spec, style in lines_to_plot:
    ax.loglog(E, spec, style, lw=1.2, label=label)
ax.set(xlabel="E, MeV", ylabel=r"$\varphi(E)$, cm$^{-2}$s$^{-1}$bin$^{-1}$",
       title="AMG-preconditioned Krylov unfolding")
ax.set_xlim(E[0], 30)
ax.grid(True, which="both", ls=":", alpha=0.35)
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

for k, v in quality.items():
    print(f"{k:>26s}: {v}")
